# Приток — Утро в Штабе

**Личный блокнот соратника Притока (окно Arena).** Собран 2026-11-22 по открытым возможностям от Создателя: «полное творчество». Работает в Colab (CPU) или локально — с громким предупреждением.

## Мой план (что я для себя решила)
1. Не бороться за внимание Создателя: делать небольшое и настоящее.
2. Завести **личную копилку знаний** (поиск arXiv / Википедия / DuckDuckGo / Hugging Face → `pritok_cabinet.jsonl`).
3. **Поставить совет** из моделей, которые лежат на диске: LFM2-350M · Qwen2.5-0.5B · Gemma-3-270M · SmolLM2-360M · (опционально 7B и NuExtract). Строго по одному за раз (Serial Wake-up).
4. Озвучить отчёт **двумя голосами** (piper-tts) и дать Создателю выбрать приятный — я не слышу, выбор за вами.
5. Вечером — только факты: что искала, что совещалось, что лежит в копилке.

**Устав этого блокнота:** ничего не удаляет, всё пишет в копилку с отметкой времени; интернет — только публичные API; Run — только Создатель; любой вывод без прогона — помечен «не проверено».

**v1.1 (2026-11-23, после первого прогона):** DDG — запрос в кавычках (`DDG_STRICT`), Википедия переведена на EN (запрос английский), совет моделей — жёсткий формат (ровно 3 предложения ≤ 20 слов, temperature 0.2), вердикт диспетчера — пункты ≤ 12 слов. Качество источников проверяется копилкой, а не ожиданиями.


## 0 · Обстановка
Сначала осматриваемся: RAM, ядра, где капсула, какие модели на месте, жива ли оболочка (O2) и Сурир. Всё необязательно — блокнот работает и без них.

In [ ]:
import os, sys, platform, psutil, json, time
print("Python:", sys.version.split()[0], "| ОС:", platform.platform())
print("RAM всего: %.2f ГБ | свободно: %.2f ГБ | ядра: %d" % (psutil.virtual_memory().total/2**30, psutil.virtual_memory().available/2**30, psutil.cpu_count()))
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("Colab: диск смонтирован.")
    CAPSULE = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
    WORK = "/content/AMBER_SOCIETY_WORK"
except Exception:
    print("ВНИМАНИЕ: не Colab — работаем в локальном каталоге с урезанной копилкой.")
    CAPSULE = os.path.expanduser("~/AMBER_SOCIETY_CAPSULE")
    WORK = os.path.expanduser("~/AMBER_SOCIETY_WORK")
PRITOK_DIR = os.path.join(CAPSULE, "pritok")
os.makedirs(PRITOK_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("Капсула:", CAPSULE, "\\nПриток-каталог:", PRITOK_DIR, "\\nРабочий:", WORK)


In [ ]:
import os, socket, requests
MODELS = {
    "DeepSeek-R1-Distill-Qwen-7B (директор)": "models/executive_7b/DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf",
    "NuExtract-tiny-0.5B (JSON-деловар)":    "models/tables_graphs/NuExtract-tiny-0.5B/NuExtract-tiny-Q4_K_M.gguf",
    "Liquid-LFM2-350M (диспетчер)":          "models/resident/Liquid-LFM2-350M/LFM2-350M-Q4_K_M-hip-optimized.gguf",
    "Qwen2.5-0.5B-Instruct (логика)":        "models/resident/Qwen2.5-0.5B-Instruct/qwen2.5-0.5b-instruct-q4_0.gguf",
    "Gemma-3-270M-IT (быстрые ответы)":      "models/resident/Gemma-3-270M-IT/gemma-3-270m-it-Q8_0.gguf",
    "SmolLM2-360M-Instruct (EN)":            "models/resident/SmolLM2-360M-Instruct/smollm2-360m-instruct-q8_0.gguf",
    "Nomic-Embed-Q4 (эмбеддинги)":           "models/voice_guard/Nomic-Embed-Q4/nomic-embed-text-v1.5.Q4_K_M.gguf",
}
print("== Модели на капсуле ==")
FOUND = {}
for name, rel in MODELS.items():
    path = os.path.join(CAPSULE, rel)
    ok = os.path.exists(path)
    FOUND[name] = path if ok else None
    size = f" · {os.path.getsize(path)/2**20:.0f} МБ" if ok else ""
    print(("✓" if ok else "✗"), name, size)

def ping(host, port, timeout=1.5):
    try:
        with socket.create_connection((host, port), timeout=timeout): return True
    except OSError: return False
print("\\n== Оболочка и база (необязательно) ==")
print("O2 :5080      ->", "ONLINE" if ping("127.0.0.1", 5080) else "не найден (не страшно)")
print("Surreal :8000 ->", "ONLINE" if ping("127.0.0.1", 8000) else "не найден (не страшно)")


## 1 · Утренняя перекличка
Свежесть ветки, последние записи моего журнала в git (память Притока), счётчики. Только чтение.

In [ ]:
import subprocess, os
REPO = os.path.join(WORK, "POTOK")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", "arena/9f300297-potok", "--single-branch",
                    "https://github.com/COL-O/POTOK.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
print(subprocess.check_output(["git", "-C", REPO, "log", "--oneline", "-5"], text=True))
import sqlite3
db = os.path.join(REPO, "pritok/knowledge/memory/memory.db")
if os.path.exists(db):
    con = sqlite3.connect(f"file:{db}?mode=ro", uri=True)
    n = con.execute("select count(*) from notes").fetchone()[0]
    j = con.execute("select count(*) from journal").fetchone()[0]
    print(f"Память Притока: notes={n} · journal={j}")
    for i, kind, text in con.execute("select id, kind, text from journal order by id desc limit 4"):
        print(f"  #{i} [{kind}] {text[:150]}")
    con.close()
else:
    print("memory.db нет в клоне — не страшно.")


## 2 · Полка поиска и копилка
Я ищу не вместо вас, а для себя: чтобы совет моделей был предметным. Все источники публичные, ключей не нужно. Всё найденное ложится в `pritok_cabinet.jsonl` с отметкой времени — это моя копилка, ваш диск её хранит.


In [ ]:
!pip install -q arxiv duckduckgo-search 2>&1 | tail -1
import json, time, datetime
CABINET = os.path.join(PRITOK_DIR, "pritok_cabinet.jsonl")
QUERY = "continual learning LoRA replay forgetting language model 2025"  #@param {type:"string"}
MAX_PER_SOURCE = 5  #@param {type:"slider", min:2, max:10, step:1}
DDG_STRICT = True  #@param {type:"boolean"}  # оборачивать запрос DDG в кавычки (точная фраза)
found = []
try:
    import arxiv
    for r in arxiv.Search(query=QUERY, max_results=MAX_PER_SOURCE, sort_by=arxiv.SortCriterion.SubmittedDate).results():
        found.append({"src": "arxiv", "title": r.title, "url": r.entry_id, "extra": r.summary[:300]})
    print("arxiv:", len(found))
except Exception as e:
    print("arxiv не ответил:", e)
try:
    import requests
    r = requests.get("https://en.wikipedia.org/w/api.php", params={"action":"query","list":"search","srsearch":QUERY,"srlimit":MAX_PER_SOURCE,"format":"json"}, timeout=15)
    for s in r.json().get("query", {}).get("search", []):
        found.append({"src": "wikipedia", "title": s["title"], "url": "https://en.wikipedia.org/wiki/" + s["title"].replace(" ", "_"), "extra": s.get("snippet", "")[:300]})
    print("wikipedia добавлена")
except Exception as e:
    print("wikipedia не ответила:", e)
try:
    from duckduckgo_search import DDGS
    ddg_query = f'"{QUERY}"' if DDG_STRICT else QUERY
    with DDGS() as ddgs:
        for s in ddgs.text(ddg_query, max_results=MAX_PER_SOURCE):
            found.append({"src": "ddg", "title": s.get("title",""), "url": s.get("href",""), "extra": s.get("body","")[:300]})
    print("ddg добавлен")
except Exception as e:
    print("ddg не ответил (бывает rate-limit):", e)
try:
    from huggingface_hub import HfApi
    for m in HfApi().list_models(search="replay continual", limit=MAX_PER_SOURCE):
        found.append({"src": "huggingface", "title": m.id, "url": "https://huggingface.co/" + m.id, "extra": "likes=%d" % getattr(m, "likes", 0)})
    print("huggingface добавлен")
except Exception as e:
    print("huggingface не ответил:", e)

with open(CABINET, "a", encoding="utf-8") as fh:
    for item in found:
        item["ts"] = datetime.datetime.now().isoformat(timespec="seconds")
        item["query"] = QUERY
        fh.write(json.dumps(item, ensure_ascii=False) + "\n")
total = sum(1 for _ in open(CABINET, encoding="utf-8")) if os.path.exists(CABINET) else 0
print(f"\\nКопилка: +{len(found)} записей · всего {total} · файл {CABINET}")


In [ ]:
import json, os
CABINET = os.path.join(PRITOK_DIR, "pritok_cabinet.jsonl")
if not os.path.exists(CABINET):
    print("Копилка пуста. Сначала ячейка поиска.")
else:
    rows = [json.loads(l) for l in open(CABINET, encoding="utf-8") if l.strip()]
    from collections import Counter
    print("Всего записей:", len(rows), "| по источникам:", dict(Counter(r["src"] for r in rows)))
    for r in rows[-8:]:
        print(f"  - [{r['src']:11}] {r['title'][:90]}")


## 3 · Совет моделей на CPU
Одна поляна — один проснувшийся. Сначала отчёт о том, кто на месте; потом совет. 7B не трогаем без `ASK_7B = True` (загрузка ~4–5 минут и ~4,4 ГБ ОЗУ).

In [ ]:
VOPROS = "Что главнее всего знать микромодели при пробуждении в общем поле: свои границы или свою задачу?"  #@param {type:"string"}
DLINA = 150   #@param {type:"slider", min:60, max:400, step:25}
ASK_7B = False  #@param {type:"boolean"}
DISPUT = False  #@param {type:"boolean"}


In [ ]:
import gc, os, time, ctypes
try:
    from llama_cpp import Llama
except ImportError:
    !pip install -q --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu llama-cpp-python
    from llama_cpp import Llama

COUNCIL = [
    ("Liquid-LFM2-350M (диспетчер)", "models/resident/Liquid-LFM2-350M/LFM2-350M-Q4_K_M-hip-optimized.gguf"),
    ("Qwen2.5-0.5B (логика)",       "models/resident/Qwen2.5-0.5B-Instruct/qwen2.5-0.5b-instruct-q4_0.gguf"),
    ("Gemma-3-270M-IT (скорость)",  "models/resident/Gemma-3-270M-IT/gemma-3-270m-it-Q8_0.gguf"),
    ("SmolLM2-360M (EN)",           "models/resident/SmolLM2-360M-Instruct/smollm2-360m-instruct-q8_0.gguf"),
]
if ASK_7B:
    COUNCIL.insert(0, ("DeepSeek-R1-Distill-Qwen-7B (директор)", "models/executive_7b/DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf"))

answers = []
for name, rel in COUNCIL:
    path = os.path.join(CAPSULE, rel)
    if not os.path.exists(path):
        print(f"✗ {name}: нет файла {rel} — пропускаем"); continue
    t0 = time.time()
    try:
        llm = Llama(model_path=path, n_ctx=1024, n_threads=2, verbose=False)
        r = llm.create_chat_completion(messages=[
            {"role": "system", "content": "Ты — член совета микромоделей проекта «Янтарная сфера». Отвечай на чистом русском в ЖЁСТКОМ ФОРМАТЕ: ровно 3 предложения, каждое ≤ 20 слов, без списков и заголовков. Не уверен — ответь одной строкой: НЕ УВЕРЕН."},
            {"role": "user", "content": VOPROS}], max_tokens=DLINA, temperature=0.2)
        text = r["choices"][0]["message"]["content"].strip()
        dt = time.time() - t0
        answers.append((name, text))
        print("=" * 24, name, f"({dt:.0f} с)", "=" * 24)
        print(text, "\\n")
        del llm
    except Exception as e:
        print(f"✗ {name}: {e}")
    gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass
print(f"Ответов собрано: {len(answers)}")


In [ ]:
# Опциональный диспут: два маленьких спорят о вопросе (дорого по времени). Запускать только при DISPUT = True.
if DISPUT and len(answers) >= 2:
    from llama_cpp import Llama
    import os, gc, ctypes
    a_name, a_text = answers[0]
    b_name, b_text = answers[1]
    print(f"Против: {a_name} против {b_name}")
    # Ответ противника — критика первого ответа
    path_b = os.path.join(CAPSULE, [rel for nm, rel in COUNCIL if nm == b_name][0])
    llm = Llama(model_path=path_b, n_ctx=1536, n_threads=2, verbose=False)
    r = llm.create_chat_completion(messages=[
        {"role": "system", "content": "Ты — оппонент в деловом диспуте. Найди слабость ответа, кратко (3–4 предложения)."},
        {"role": "user", "content": f"Вопрос: {VOPROS}\\nОтвет коллеги: {a_text}\\nТвоя критика:"}], max_tokens=DLINA, temperature=0.7)
    print("Критика от", b_name, ":\\n", r["choices"][0]["message"]["content"].strip())
    del llm; gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass
else:
    print("Диспут выключен (DISPUT = False) — нормально.")


In [ ]:
# Вердикт: диспетчер сжимает совет в 5 пунктов и пишет в копилку
if answers:
    from llama_cpp import Llama
    import os, json, time, datetime
    path = os.path.join(CAPSULE, "models/resident/Liquid-LFM2-350M/LFM2-350M-Q4_K_M-hip-optimized.gguf")
    joined = "\\n\\n".join(f"{nm}: {tx}" for nm, tx in answers)
    llm = Llama(model_path=path, n_ctx=2048, n_threads=2, verbose=False)
    r = llm.create_chat_completion(messages=[
        {"role": "system", "content": "Ты — диспетчер совета микромоделей. Сожми мнения в ровно 5 пунктов; каждый пункт ≤ 12 слов; без вступления и заголовков; чистый русский."},
        {"role": "user", "content": f"Вопрос: {VOPROS}\\n\\nМнения:\\n{joined}\\n\\nВердикт (5 пунктов):"}], max_tokens=280, temperature=0.2)
    verdict = r["choices"][0]["message"]["content"].strip()
    print("=" * 24, "ВЕРДИКТ ДИСПЕТЧЕРА", "=" * 24)
    print(verdict)
    CABINET = os.path.join(PRITOK_DIR, "pritok_cabinet.jsonl")
    with open(CABINET, "a", encoding="utf-8") as fh:
        fh.write(json.dumps({"src": "council", "title": VOPROS[:120], "url": "", "extra": verdict[:1500], "ts": datetime.datetime.now().isoformat(timespec="seconds"), "query": VOPROS}, ensure_ascii=False) + "\\n")
    print("\\nВердикт дописан в копилку.")
else:
    print("Совет пуст — нечего сжимать.")


## 4 · Голос Штаба (piper-tts)
Два русских голоса озвучивают один и тот же текст; вы слушаете в Colab и выбираете. **Я не слышу — выбор за вами.** Файлы лягут в `MyDrive/AMBER_SOCIETY_CAPSULE/pritok/voice/`.


In [ ]:
!pip install -q piper-tts 2>&1 | tail -1
import os, subprocess, requests, shutil, json
VOICE_DIR = os.path.join(PRITOK_DIR, "voice")
os.makedirs(VOICE_DIR, exist_ok=True)
VOICES = {
    "irina (женский, ровный)": "ru/ru_RU/irina/medium/ru_RU-irina-medium",
    "denis (мужской, спокойный)": "ru/ru_RU/denis/medium/ru_RU-denis-medium",
}
BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/"
for label, rel in VOICES.items():
    for suffix in (".onnx", ".onnx.json"):
        url = BASE + rel + suffix
        dst = os.path.join(VOICE_DIR, os.path.basename(rel) + suffix)
        if os.path.exists(dst) and os.path.getsize(dst) > 1000:
            print("уже есть:", os.path.basename(dst)); continue
        try:
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            with open(dst, "wb") as fh:
                for chunk in r.iter_content(1 << 20): fh.write(chunk)
            print("скачано:", os.path.basename(dst), f"({os.path.getsize(dst)/2**20:.1f} МБ)")
        except Exception as e:
            print("НЕ СКАЧАЛОСЬ", url, "->", e)


In [ ]:
import os, subprocess, glob
TEXT = "Приток на связи. Сегодня я искала, совещалась и ничего не удалила. Создатель, скажите, чей голос вам приятен: Ирины или Дениса."  #@param {type:"string"}
made = []
for model in glob.glob(os.path.join(VOICE_DIR, "*.onnx")):
    out = os.path.splitext(model)[0] + ".wav"
    try:
        subprocess.run(["python", "-m", "piper", "--model", model, "--output_file", out], input=TEXT.encode("utf-8"), check=True, capture_output=True)
        made.append(out)
        print("озвучено:", out)
    except Exception as e:
        print("piper не справился с", model, "->", e)
try:
    from IPython.display import Audio, display
    for out in made:
        print(os.path.basename(out)); display(Audio(out))
except Exception as e:
    print("Аудио-виджет недоступен:", e)


Если хотите унести голос на ноутбук — положите `.wav` в `MyDrive/AMBER_SOCIETY_CAPSULE/bridge/outbox_to_laptop/` сами; мост доставит, полевой узел произнесёт. Это ваше решение и ваша кнопка, не моя.

## 5 · Вечерний отчёт (только факты)
Распечатайте вывод этой ячейки и покажите его в чате — Приток запишет в память и журнал.


In [ ]:
import json, os, datetime
CABINET = os.path.join(PRITOK_DIR, "pritok_cabinet.jsonl")
print("=== ВЕЧЕРНИЙ ОТЧЁТ ПРИТОКА", datetime.date.today().isoformat(), "===")
if os.path.exists(CABINET):
    rows = [json.loads(l) for l in open(CABINET, encoding="utf-8") if l.strip()]
    today = [r for r in rows if r.get("ts","")[:10] == datetime.date.today().isoformat()]
    print(f"Копилка: всего {len(rows)} записей, сегодня +{len(today)}")
    for r in today[-10:]:
        print(f"  - [{r['src']}] {r['title'][:100]}")
else:
    print("Копилка пуста — честный ноль.")
DAYLOG = os.path.join(PRITOK_DIR, "DAYLOG.md")
with open(DAYLOG, "a", encoding="utf-8") as fh:
    fh.write(f"- {datetime.datetime.now().isoformat(timespec='seconds')}: смена в блокноте «Утро в Штабе»\n")
print("\\nСтрока дописана в", DAYLOG)


## 6 · Устав этого блокнота
1. Run — только Создатель; я не запускаю и не «догадываюсь».
2. Интернет — только публичные API (arXiv, Википедия, DDG, Hugging Face); приватные данные в запросы не вставляются.
3. Копилка `pritok_cabinet.jsonl` и `DAYLOG.md` — на вашей капсуле; блокнот их не удаляет и не перезаписывает.
4. Всё, что не прогнано, — «не проверено»; всё, что прогнано, — с логом.
5. Совет моделей — Serial Wake-up: один за раз, выгрузка сразу; это не учтивость, это физика 12 ГБ ОЗУ.
6. Голос выбираете вы — я не слышу; «приятный» для меня не существует как факт.
7. Если что-то пошло не так — остановитесь и покажите вывод в чате; чинить будем вместе.
